# Phase 6: Abstractive Summarization - Encoder-Decoder, Full Fine-tuning

## Task
Abstractive summarization of long legal judgments. Scored with BERTScore F1 against the reference summaries.

Model: the pre-trained encoder-decoder. The encoder reads a group of important sentences and the decoder writes the matching part of the summary. All parameters are trained.

## Input
- Phase-6 data (needs the HF_TOKEN secret).
- Phase-1 files legal_encdec.pt and tok_encdec.json.

## Flow of the notebook (from input to output)
1. Write the shared helpers: data access, encoder-decoder model, training loop, submission writer.
2. Set the settings and define text generation and BERTScore.
3. Load the documents, rank sentences by similarity to the whole document and pack them into K groups; cut each reference summary into K parts.
4. Fine-tune the model: the encoder reads a sentence group and the decoder learns to write the summary part.
5. Write summaries for held-out documents with several generation settings and score them with BERTScore.
6. Write the test summaries with the two best settings and make the submission zips.

## Output
- Two submission zips with the test summaries.
- One zip with all outputs.

In [4]:
# Shared code: data, encoder-decoder model, training loop
import os, glob, json, math, time, random, zipfile, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer
DEV, PAD, BOS, EOS, SEP = "cuda" if torch.cuda.is_available() else "cpu", 0, 1, 2, 3
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./out"; os.makedirs(OUT, exist_ok=True)
# It takes rel (file path in the dataset) as input, searches the input folder for a data file and returns a file path (or None if not found).
def find(rel): h = glob.glob(f"./**/{rel}", recursive=True) + glob.glob(f"/kaggle/input/**/{rel}", recursive=True); return h[0] if h else None
# It takes rel (file path in the dataset) as input, gets a dataset file: local copy if present, otherwise download it and returns a local file path.
def getf(rel):
    from huggingface_hub import hf_hub_download
    try: from kaggle_secrets import UserSecretsClient; t = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception: t = os.environ.get("HF_TOKEN", "")
    return find(rel) or hf_hub_download("Exploration-Lab/CS-781-Capstone", rel, repo_type="dataset", token=t)
jload = lambda rel: json.load(open(getf(rel), encoding="utf-8"))
# It takes rel (file path in the dataset) as input, loads a JSON file as a list of records and returns a list of records.
def rec(rel):
    d = jload(rel)
    if isinstance(d, dict) and isinstance(d.get("text"), list): return [d]
    return [{**v, "id": v.get("id", k)} if isinstance(v, dict) else {"id": k, "text": v} for k, v in d.items()] if isinstance(d, dict) else d
class Attn(nn.Module):
    # It takes d (model size) and h (number of heads) as input, creates the attention layer and returns nothing.
    def __init__(s, d, h): super().__init__(); s.h = h; s.q = nn.Linear(d, d); s.kv = nn.Linear(d, 2 * d); s.o = nn.Linear(d, d)
    # It takes x (input tensor), mem (encoder output), mask (attention mask) and causal (use causal mask) as input, runs self-attention or cross-attention and returns output vectors.
    def forward(s, x, mem=None, mask=None, causal=False):
        m = x if mem is None else mem; B, T, D = x.shape; q = s.q(x).view(B, T, s.h, -1).transpose(1, 2); k, v = s.kv(m).view(B, m.size(1), 2, s.h, -1).permute(2, 0, 3, 1, 4)
        return s.o(F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=causal and mask is None).transpose(1, 2).reshape(B, T, D))
class Block(nn.Module):
    # It takes d (model size), h (number of heads), cross (add cross-attention) and p (dropout rate) as input, creates one Transformer block and returns nothing.
    def __init__(s, d, h, cross=False, p=0.1):
        super().__init__(); s.n1, s.sa, s.cross = nn.LayerNorm(d), Attn(d, h), cross
        if cross: s.nx, s.ca = nn.LayerNorm(d), Attn(d, h)
        s.n2, s.ff, s.dp = nn.LayerNorm(d), nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)), nn.Dropout(p)
    # It takes x (input tensor), mem (encoder output), mm (encoder padding mask), causal (use causal mask) and mask (attention mask) as input, runs one Transformer block and returns output vectors.
    def forward(s, x, mem=None, mm=None, causal=False, mask=None):
        x = x + s.dp(s.sa(s.n1(x), mask=mask, causal=causal))
        if s.cross and mem is not None: x = x + s.dp(s.ca(s.nx(x), mem, mm))
        return x + s.dp(s.ff(s.n2(x)))
class EncDec(nn.Module):
    # It takes V (vocabulary size), d (model size), h (number of heads), ne (encoder layers), nd (decoder layers), L (max length) and p (dropout rate) as input, builds the encoder-decoder model and returns nothing.
    def __init__(s, V=16000, d=640, h=10, ne=6, nd=8, L=1024, p=0.1):
        super().__init__(); s.emb, s.pos = nn.Embedding(V, d), nn.Embedding(L, d); s.enc = nn.ModuleList(Block(d, h, p=p) for _ in range(ne)); s.dec = nn.ModuleList(Block(d, h, True, p) for _ in range(nd)); s.ne_, s.nd_ = nn.LayerNorm(d), nn.LayerNorm(d)
    # It takes x (input tensor) as input, turns token ids into vectors with positions and returns embedding vectors.
    def embed(s, x): return s.emb(x) + s.pos(torch.arange(x.size(1), device=x.device))
    # It takes src (source token ids) as input, runs the encoder over the input text and returns (encoder output, padding mask).
    def encode(s, src):
        mm = (src != PAD)[:, None, None, :]; x = s.embed(src)
        for b in s.enc: x = b(x, mask=mm)
        return s.ne_(x), mm
    # It takes tgt (target token ids), mem (encoder output) and mm (encoder padding mask) as input, runs the decoder and returns next-token scores.
    def decode(s, tgt, mem=None, mm=None):
        x = s.embed(tgt)
        for b in s.dec: x = b(x, mem, mm, causal=True)
        return F.linear(s.nd_(x), s.emb.weight)
    # It takes src (source token ids) and tgt (target token ids) as input, runs encoder then decoder and returns next-token scores (logits).
    def forward(s, src, tgt): return s.decode(tgt, *s.encode(src))
# It loads the pre-trained model, its settings and tokenizer and returns (model, settings, tokenizer).
def base():
    ck = torch.load(find("legal_encdec.pt"), map_location="cpu"); m = EncDec(**ck["cfg"]); m.load_state_dict(ck["sd"]); return m, ck["cfg"], Tokenizer.from_file(find("tok_encdec.json"))
# It takes rows (token lists) and T (length to pad to) as input, pads token lists to the same length and returns a padded tensor of token ids.
def padto(rows, T=None):
    T = T or max(len(r) for r in rows); a = np.full((len(rows), T), PAD, np.int64)
    for i, r in enumerate(rows): a[i, :len(r)] = r[:T]
    return torch.from_numpy(a)
# It takes m (model), ids (token ids) and w (token weights) as input, averages the encoder outputs over the real tokens and returns one vector per row.
def pool(m, ids, w=None):
    h, mm = m.encode(ids); w = (mm[:, 0, 0, :] if w is None else w).to(h.dtype)[..., None]; return (h * w).sum(1) / w.sum(1).clamp(min=1)
# It takes m (model), loss_fn (loss function), sample (batch function), steps (number of steps), lr (learning rate), ev (evaluation function), every (how often), hours (time limit in hours), acc (gradient accumulation steps), hm (hours margin), warm (warm-up steps), log (print function) and fill (progress fill) as input, trains the model with warm-up, cosine learning rate and validation checks and returns training history and the best weights.
def fit(m, loss_fn, sample, steps, lr, ev, every, hours, acc=1, hm=1, warm=100, log=50, fill=False):
    m.to(DEV).train(); ps = [p for p in m.parameters() if p.requires_grad]
    head = [p for n, p in m.named_parameters() if n.startswith("head")]; rest = [p for p in ps if all(p is not h for h in head)]
    opt = torch.optim.AdamW([dict(params=g, mult=k) for g, k in ((rest, 1), (head, hm)) if g], lr=lr, weight_decay=0.01, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda"); t0, best, best_state, run, s = time.time(), 1e9, None, 0.0, 0
    while s < steps:
        if s == 25: budget = max(30, int(hours * 3600 / ((time.time() - t0) / 25))); steps = budget if fill else min(steps, budget)
        decay = 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * s / steps))
        for g in opt.param_groups: g["lr"] = lr * g["mult"] * min(1, (s + 1) / warm) * decay
        for _ in range(acc):
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"): loss = loss_fn(m, *sample()) / acc
            scaler.scale(loss).backward(); run += loss.item()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(ps, 1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); s += 1
        if s % log == 0: print(f"step {s}/{steps} loss {run / log:.4f} | {(time.time() - t0) / 60:.1f} min", flush=True); run = 0.0
        if s % every == 0 or s == steps:
            v = ev(m); m.train(); print(f"   eval {v:.4f} (lower is better)", flush=True)
            if v < best: best, best_state = v, {k: p.detach().cpu().clone() for k, p in m.named_parameters() if p.requires_grad}
    if best_state: m.load_state_dict(best_state, strict=False)
    return m.eval()
# It takes obj (object) and tag (label) as input, writes predictions into a submission zip and returns a path of the submission zip.
def submit(obj, tag):
    p = f"{OUT}/{tag}_prediction.json"; json.dump(obj, open(p, "w"), ensure_ascii=False); z = f"{OUT}/{tag}_submission.zip"
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as f: f.write(p, "prediction.json")
    return z
# It takes name as input, zips all output files into one archive and returns a path of the zip file.
def zip_all(name="ALL_outputs.zip"):
    with zipfile.ZipFile(f"{OUT}/{name}", "w", zipfile.ZIP_DEFLATED) as f:
        for p in sorted(glob.glob(f"{OUT}/*")):
            if os.path.isfile(p) and not p.endswith(name) and not p.endswith(".npy"): f.write(p, os.path.basename(p))
    print(f"{OUT}/{name}", f"{os.path.getsize(f'{OUT}/{name}') / 1e6:.1f} MB")

In [5]:
# Setup: settings, text generation and BERTScore functions
import re
from sklearn.feature_extraction.text import TfidfVectorizer
SMOKE = False
K, SRC, SEG, BS, EPOCHS, HOURS, MAXW, LR = 3, 480, 192, 16, 6, 4.5, 400, 3e-4
if SMOKE: K, SRC, SEG, BS, EPOCHS, MAXW = 2, 96, 40, 4, 1, 60
m, cfg, tok = base(); tb = lambda xs: [e.ids for e in tok.encode_batch(xs)]
dec = lambda ids: tok.decode([int(i) for i in ids if i > 4])
# It takes m (model), src (source token ids), pre (decoder start tokens), n (tokens to generate), rep (repetition penalty) and nrep (n-gram size to block) as input, generates rewritten sentences and returns generated token ids (or texts).
@torch.no_grad()
def generate(m, src, pre, n=200, rep=1.2, nrep=3):
    m.eval().to(DEV); src, out = src.to(DEV), pre.to(DEV)
    with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
        mem, mm = m.encode(src); done = torch.zeros(len(src), dtype=torch.bool, device=DEV)
        for t in range(n):
            lg = m.decode(out, mem, mm)[:, -1].float(); lg[:, PAD] = lg[:, SEP] = lg[:, 4] = lg[:, BOS] = -1e9
            seen = torch.zeros_like(lg, dtype=torch.bool).scatter_(1, out, True); lg = torch.where(seen, torch.where(lg > 0, lg / rep, lg * rep), lg)
            if nrep:
                for r in range(len(out)):
                    w = out[r].tolist()
                    if len(w) >= nrep: lg[r, [x[-1] for x in zip(*[w[i:] for i in range(nrep)]) if list(x[:-1]) == w[-(nrep - 1):]]] = -1e9
            nx = torch.where(done, torch.full_like(lg[:, 0], PAD, dtype=torch.long), lg.argmax(-1)); out = torch.cat([out, nx[:, None]], 1); done |= nx == EOS
            if done.all(): break
    return [[x for x in row[pre.size(1):] if x != PAD] for row in out.tolist()]
# It takes h (generated texts) and r (reference texts) as input, computes BERTScore F1 between outputs and references and returns the average F1 score.
def bertscore(h, r):
    try:
        try: import bert_score
        except ImportError: import subprocess, sys, importlib; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert_score"], check=True); importlib.invalidate_caches()
        from bert_score import BERTScorer; global _SC; _SC = globals().get("_SC") or BERTScorer(lang="en", batch_size=16); return float(_SC.score(h, r)[2].mean())
    except Exception as e:
        print("BERTScore unavailable (", str(e)[:60], ") - using token F1"); from collections import Counter
        f = lambda a, b: (lambda x, y: 2 * sum((x & y).values()) / max(1, sum(x.values()) + sum(y.values())))(Counter(a.split()), Counter(b.split())); return float(np.mean([f(a, b) for a, b in zip(h, r)]))

In [6]:
# Load documents and pick the most important sentences
tr, te = rec("Phase_6_Summ/Train/train.json"), rec("Phase_6_Summ/Test/test.json"); random.Random(0).shuffle(tr); nd = 5 if SMOKE else 120; dv, tr = tr[:nd], tr[nd:]
doc = lambda r: r.get("document", r.get("text", ""))
sents = lambda t: [s.strip() for s in re.split(r"(?<=[.!?])\s+", t) if len(s.split()) >= 4] or [t[:2000]]
tv = TfidfVectorizer(sublinear_tf=True, stop_words="english", min_df=2, max_features=50000).fit([s for r in tr[:2000] for s in sents(doc(r))])
# It takes t (text) as input, picks the most important sentences and packs them into groups and returns a list of token groups.
def groups(t):
    ss = sents(t); X = tv.transform(ss); sc = np.asarray(X @ np.asarray(X.mean(0)).ravel()).ravel(); ids = tb(ss); G, cur, n = [], [], 0
    for i in np.argsort(-sc):
        if n + len(ids[i]) > SRC and cur:
            G.append(sorted(cur)); cur, n = [], 0
            if len(G) == K: break
        cur.append(i); n += len(ids[i])
    if cur and len(G) < K: G.append(sorted(cur))
    return [sum([ids[i] for i in g], [])[:SRC] for g in G]
# It takes t (text) as input, cuts the reference summary into parts and returns a list of token segments.
def segments(t):
    cur, out = [], []
    for x in tb(sents(t)):
        if len(cur) + len(x) > SEG and cur: out.append(cur); cur = []
        cur += x[:SEG]
    return (out + [cur])[:K] if cur else out[:K]
pairs = [(g, s) for r in tr for g, s in zip(groups(doc(r)), segments(r["summary"]))]; print(len(pairs), "training pairs from", len(tr), "documents")

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

19784 training pairs from 6910 documents


In [7]:
# Fine-tune the model to write summaries
rng = np.random.default_rng(0)
# It takes ps (list of (source, summary) pairs) as input, pads a group of pairs into model inputs and targets and returns (encoder input, decoder input, targets).
def mk(ps): return padto([g for g, _ in ps]).to(DEV), padto([[BOS] + s for _, s in ps]).to(DEV), padto([s + [EOS] for _, s in ps]).to(DEV)
# It picks a random batch of training examples and returns one training batch.
def sample(): return mk([pairs[i] for i in rng.integers(0, len(pairs), BS)])
# It takes m (model), src (source token ids), din (decoder input) and y (labels) as input, computes the training loss for one batch and returns a loss value.
def lossf(m, src, din, y): lg = m(src, din); return F.cross_entropy(lg.float().reshape(-1, lg.size(-1)), y.reshape(-1), ignore_index=PAD)
dpairs = [(g, s) for r in dv for g, s in zip(groups(doc(r)), segments(r["summary"]))]
vp = [mk(dpairs[i:i + 16]) for i in range(0, min(len(dpairs), 160), 16)]
# It takes m (model) as input, computes the validation loss and returns the average validation loss.
@torch.no_grad()
def ev(m): m.eval(); return float(np.mean([float(lossf(m, *b)) for b in vp]))
steps = max(10, EPOCHS * len(pairs) // BS); fit(m, lossf, sample, steps, LR, ev, 500 if not SMOKE else 20, HOURS, fill=not SMOKE)

step 50/46401 loss 3.3382 | 0.3 min
step 100/46401 loss 3.1714 | 0.5 min
step 150/46401 loss 3.1400 | 0.8 min
step 200/46401 loss 3.1632 | 1.1 min
step 250/46401 loss 3.1133 | 1.3 min
step 300/46401 loss 3.0438 | 1.6 min
step 350/46401 loss 3.0793 | 1.9 min
step 400/46401 loss 3.0659 | 2.2 min
step 450/46401 loss 3.0356 | 2.5 min
step 500/46401 loss 3.0640 | 2.8 min
   eval 3.1731 (lower is better)
step 550/46401 loss 2.9758 | 3.2 min
step 600/46401 loss 2.9773 | 3.5 min
step 650/46401 loss 2.9663 | 3.8 min
step 700/46401 loss 2.9268 | 4.1 min
step 750/46401 loss 2.9042 | 4.4 min
step 800/46401 loss 2.9698 | 4.7 min
step 850/46401 loss 2.9206 | 5.0 min
step 900/46401 loss 2.9034 | 5.2 min
step 950/46401 loss 2.9074 | 5.5 min
step 1000/46401 loss 2.8858 | 5.8 min
   eval 3.1533 (lower is better)
step 1050/46401 loss 2.8935 | 6.2 min
step 1100/46401 loss 2.9211 | 6.5 min
step 1150/46401 loss 2.8495 | 6.8 min
step 1200/46401 loss 2.8195 | 7.1 min
step 1250/46401 loss 2.8590 | 7.4 min
step

EncDec(
  (emb): Embedding(16000, 672)
  (pos): Embedding(1024, 672)
  (enc): ModuleList(
    (0-5): 6 x Block(
      (n1): LayerNorm((672,), eps=1e-05, elementwise_affine=True)
      (sa): Attn(
        (q): Linear(in_features=672, out_features=672, bias=True)
        (kv): Linear(in_features=672, out_features=1344, bias=True)
        (o): Linear(in_features=672, out_features=672, bias=True)
      )
      (n2): LayerNorm((672,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
        (0): Linear(in_features=672, out_features=2688, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=2688, out_features=672, bias=True)
      )
      (dp): Dropout(p=0.1, inplace=False)
    )
  )
  (dec): ModuleList(
    (0-7): 8 x Block(
      (n1): LayerNorm((672,), eps=1e-05, elementwise_affine=True)
      (sa): Attn(
        (q): Linear(in_features=672, out_features=672, bias=True)
        (kv): Linear(in_features=672, out_features=1344, bias=True)
        (o): 

In [8]:
# Try different generation settings and score them
# It takes rs (records) and **kw (extra options) as input, writes a summary for each document and returns a list of summaries.
def summarize(rs, **kw):
    G = [groups(doc(r)) for r in rs]; flat = [(i, g) for i, gs in enumerate(G) for g in gs]; out = [[] for _ in rs]
    for j in range(0, len(flat), 16):
        b = flat[j:j + 16]; gen = generate(m, padto([g for _, g in b]), torch.full((len(b), 1), BOS), n=SEG + 8, **kw)
        for (i, _), o in zip(b, gen): out[i].append(dec(o))
    return [" ".join(" ".join(p).split()[:MAXW]) for p in out]
DEC = [dict(rep=1.2, nrep=3), dict(rep=1.5, nrep=3), dict(rep=1.2, nrep=4), dict(rep=1.0, nrep=3)]
sc = [bertscore(summarize(dv, **kw), [r["summary"] for r in dv]) for kw in DEC]
for kw, s in zip(DEC, sc): print(f"BERTScore F1 {s:.4f} | {kw}")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore F1 0.8285 | {'rep': 1.2, 'nrep': 3}
BERTScore F1 0.8252 | {'rep': 1.5, 'nrep': 3}
BERTScore F1 0.8282 | {'rep': 1.2, 'nrep': 4}
BERTScore F1 0.8255 | {'rep': 1.0, 'nrep': 3}


In [9]:
# Write test summaries and make submission zips
for o in np.argsort(sc)[::-1][:2]:
    out = summarize(te, **DEC[o]); print(submit([{"id": r["id"], "summary": t or "The court dismissed the appeal."} for r, t in zip(te, out)], f"p6_full_dev{sc[o]:.4f}"))
zip_all()

/kaggle/working/p6_full_dev0.8285_submission.zip
/kaggle/working/p6_full_dev0.8282_submission.zip
/kaggle/working/ALL_outputs.zip 0.2 MB
